# Phần 1: Tải và Khám phá dữ liệu (EDA)


In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Đọc dữ liệu từ thư mục data
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

print(f"Kích thước tập Train: {train.shape}")
print(f"Kích thước tập Test: {test.shape}")

# Lưu lại Id để nộp bài Kaggle
train_ID = train['Id']
test_ID = test['Id']

# Loại bỏ cột Id vì không mang ý nghĩa dự báo
train.drop("Id", axis=1, inplace=True)
test.drop("Id", axis=1, inplace=True)

Kích thước tập Train: (1460, 81)
Kích thước tập Test: (1459, 80)


# Phần 2: Xử lý ngoại lệ (Outliers)
Loại bỏ các điểm dữ liệu nhiễu (ngoại lệ cực đoan) có diện tích rất lớn nhưng giá thấp theo tài liệu gốc của bộ dữ liệu Ames Housing, đồng thời biến đổi logarit cho biến mục tiêu `SalePrice` để chuẩn hóa phân phối.

In [12]:
# Loại bỏ 2 điểm ngoại lệ theo nghiên cứu gốc
train = train.drop(train[(train['GrLivArea'] > 4000) & (train['SalePrice'] < 300000)].index)

# Tách biến mục tiêu SalePrice và áp dụng logarit (log1p)
y_train = np.log1p(train['SalePrice'].reset_index(drop=True))
train.drop(['SalePrice'], axis=1, inplace=True)

# Gộp chung train và test để xử lý tiền xử lý đồng nhất (tránh rò rỉ dữ liệu)
ntrain = train.shape[0]
ntest = test.shape[0]
all_data = pd.concat((train, test)).reset_index(drop=True)

# Phần 3: Tiền xử lý và Điền khuyết dữ liệu (Imputation)
Xử lý các giá trị khuyết (Missing values) dựa trên đặc thù nghiệp vụ: điền `'None'` hoặc `0` cho các cấu trúc không tồn tại, dùng trung vị theo khu phố cho chiều dài mặt tiền (`LotFrontage`), và điền Mode cho các biến định danh.

In [13]:
# Điền 'None' cho các biến khuyết mang nghĩa 'Không có cấu trúc này'
none_cols = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu', 
             'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
             'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2']
for col in none_cols:
    all_data[col] = all_data[col].fillna('None')

# Điền 0 cho các số đo diện tích khi cấu trúc không tồn tại
zero_cols = ['GarageYrBlt', 'GarageArea', 'GarageCars', 
             'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 
             'BsmtFullBath', 'BsmtHalfBath', 'MasVnrArea']
for col in zero_cols:
    all_data[col] = all_data[col].fillna(0)

# Điền LotFrontage theo giá trị trung vị của từng khu phố (Neighborhood)
all_data['LotFrontage'] = all_data.groupby('Neighborhood')['LotFrontage'].transform(
    lambda x: x.fillna(x.median())
)

# Điền Mode cho các cột khuyết số lượng ít còn lại
mode_cols = ['MSZoning', 'Electrical', 'KitchenQual', 'Exterior1st', 'Exterior2nd', 'SaleType', 'Utilities']
for col in mode_cols:
    all_data[col] = all_data[col].fillna(all_data[col].mode()[0])

all_data['Functional'] = all_data['Functional'].fillna('Typ')

# Phần 4: Kỹ thuật tạo đặc trưng (Feature Engineering)
Xây dựng các siêu biến (Super features) như tổng diện tích, tổng số phòng tắm, tuổi thọ căn nhà và các cờ nhị phân giúp mô hình học tập tốt hơn.

In [14]:
# Tạo tổng diện tích sinh hoạt
all_data['TotalSF'] = all_data['TotalBsmtSF'] + all_data['1stFlrSF'] + all_data['2ndFlrSF']

# Quy đổi tổng số phòng tắm
all_data['Total_Bathrooms'] = (all_data['FullBath'] + (0.5 * all_data['HalfBath']) +
                               all_data['BsmtFullBath'] + (0.5 * all_data['BsmtHalfBath']))

# Tính tuổi thọ căn nhà và thời điểm trùng tu
all_data['HouseAge'] = all_data['YrSold'] - all_data['YearBuilt']
all_data['RemodAge'] = all_data['YrSold'] - all_data['YearRemodAdd']
all_data['IsRemodeled'] = (all_data['YearRemodAdd'] != all_data['YearBuilt']).astype(int)
all_data['HasPool'] = all_data['PoolArea'].apply(lambda x: 1 if x > 0 else 0)
all_data['Has2ndFloor'] = all_data['2ndFlrSF'].apply(lambda x: 1 if x > 0 else 0)
all_data['HasGarage'] = all_data['GarageArea'].apply(lambda x: 1 if x > 0 else 0)
all_data['HasBsmt'] = all_data['TotalBsmtSF'].apply(lambda x: 1 if x > 0 else 0)
all_data['HasFireplace'] = all_data['Fireplaces'].apply(lambda x: 1 if x > 0 else 0)

# Phần 5: Xử lý độ lệch (Box-Cox) & Mã hóa biến phân loại (Encoding)
Sử dụng hàm Box-Cox để nắn chỉnh độ lệch của các biến số học, sau đó chuyển đổi các biến dạng chữ (categorical) thành dạng số bằng One-Hot Encoding.

In [15]:
from scipy.stats import skew
from scipy.special import boxcox1p

# Lấy danh sách TÊN CỘT của các biến số học (thay vì lấy index dạng số nguyên)
numeric_feats = all_data.select_dtypes(include=[np.number]).columns

# Tính độ lệch cho từng cột số dựa vào tên cột
skewed_feats = all_data[numeric_feats].apply(lambda x: skew(x.dropna())).sort_values(ascending=False)

# Lọc các biến có độ lệch mạnh (abs(skew) > 0.75)
skewness = pd.DataFrame({'Skew': skewed_feats})
skewness = skewness[abs(skewness['Skew']) > 0.75]

# Áp dụng biến đổi Box-Cox1p cho các biến lệch mạnh
lam = 0.15
for feat in skewness.index:
    all_data[feat] = boxcox1p(all_data[feat], lam)

# Mã hóa One-Hot Encoding cho toàn bộ các biến phân loại còn lại
all_data = pd.get_dummies(all_data)
print(f"Kích thước tập dữ liệu sau khi One-Hot Encoding: {all_data.shape}")

# Tách lại thành tập Train và Test hoàn chỉnh
X_train = all_data[:ntrain]
X_test = all_data[ntrain:]

Kích thước tập dữ liệu sau khi One-Hot Encoding: (2917, 310)


# Phần 6: Thiết lập Kiểm chứng chéo và Huấn luyện mô hình đơn lẻ
Cài đặt hàm đánh giá Cross-Validation (K-Fold) và khởi tạo các mô hình cơ sở (Lasso, ElasticNet, Gradient Boosting, XGBoost, LightGBM).

In [16]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import ElasticNet, Lasso, RidgeCV
from sklearn.ensemble import GradientBoostingRegressor
import xgboost as xgb
import lightgbm as lgb

n_folds = 5

def rmsle_cv(model):
    kf = KFold(n_folds, shuffle=True, random_state=42).get_n_splits(X_train.values)
    rmse = np.sqrt(-cross_val_score(model, X_train.values, y_train, 
                                    scoring="neg_mean_squared_error", cv = kf))
    return(rmse)

# Khởi tạo mô hình với các siêu tham số tối ưu
lasso = make_pipeline(RobustScaler(), Lasso(alpha=0.0005, random_state=1))
enet = make_pipeline(RobustScaler(), ElasticNet(alpha=0.0005, l1_ratio=0.9, random_state=3))

gboost = GradientBoostingRegressor(n_estimators=3000, learning_rate=0.05,
                                   max_depth=4, max_features='sqrt',
                                   min_samples_leaf=15, min_samples_split=10, 
                                   loss='huber', random_state=5)

model_xgb = xgb.XGBRegressor(colsample_bytree=0.4603, gamma=0.0468, 
                             learning_rate=0.01, max_depth=3, 
                             min_child_weight=1.7817, n_estimators=3460,
                             reg_alpha=0.4640, reg_lambda=0.8571,
                             subsample=0.5213, random_state=7, n_jobs=-1)

model_lgb = lgb.LGBMRegressor(objective='regression', num_leaves=5,
                              learning_rate=0.01, n_estimators=3000,
                              max_bin=55, bagging_fraction=0.8,
                              bagging_freq=5, feature_fraction=0.2319,
                              feature_fraction_seed=9, bagging_seed=9,
                              verbose=-1)

# Kiểm tra điểm số CV RMSE
print("Lasso score: {:.4f}".format(np.mean(rmsle_cv(lasso))))
print("XGBoost score: {:.4f}".format(np.mean(rmsle_cv(model_xgb))))

c:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.367867e-01, tolerance: 1.891e-02
  model = cd_fast.enet_coordinate_descent(
c:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.120321e-02, tolerance: 1.837e-02
  model = cd_fast.enet_coordinate_descent(
c:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, chec

Lasso score: 0.1092
XGBoost score: 0.1148


# Phần 7: Xây dựng Stacking Regressor và Xuất kết quả
Xây dựng mô hình học kết hợp đa tầng (Stacking Ensemble) để tối ưu hóa độ chính xác và xuất file kết quả nộp bài vào thư mục report.

In [ ]:
from mlxtend.regressor import StackingCVRegressor

# Thiết lập Stacking Regressor
stack_gen = StackingCVRegressor(regressors=(lasso, enet, gboost, model_xgb, model_lgb),
                                meta_regressor=lasso,
                                use_features_in_secondary=True)

# Huấn luyện mô hình
print("Đang huấn luyện mô hình Stacking...")
stack_gen.fit(X_train.values, y_train)

# Định nghĩa hàm dự đoán kết hợp trọng số (Weighted Blend)
def blended_predictions(X):
    return ((0.1 * lasso.predict(X)) + \
            (0.1 * enet.predict(X)) + \
            (0.1 * gboost.predict(X)) + \
            (0.15 * model_xgb.predict(X)) + \
            (0.1 * model_lgb.predict(X)) + \
            (0.45 * stack_gen.predict(np.array(X))))

# Tạo file dự đoán cuối cùng
final_preds = np.expm1(blended_predictions(X_test.values))

sub = pd.DataFrame()
sub['Id'] = test_ID
sub['SalePrice'] = final_preds
sub.to_csv('report/submission_stacked.csv', index=False)
print("Đã hoàn tất! File kết quả lưu tại: report/submission_stacked.csv")

Đang huấn luyện mô hình Stacking...


c:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.255810e-01, tolerance: 1.838e-02
  model = cd_fast.enet_coordinate_descent(
c:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.186069e-02, tolerance: 1.917e-02
  model = cd_fast.enet_coordinate_descent(
c:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, chec